# Smart Meter Electricity Consumption Data Pipeline

## Silver Layer - Cleansing & Standardization

### Objective

The Silver layer transforms the raw Bronze data into clean, validated, and standardized data for analytics.

In this notebook we will:

- Read Bronze Delta Tables
- Remove duplicate records
- Handle null and invalid values
- Standardize timestamps
- Join household information
- Create validation flags
- Save the cleaned data as a Silver Delta Table

Technology Used:
- PySpark
- Delta Lake
- Databricks

In [0]:
bronze_meter_df=spark.table("workspace.default.bronze_meter_readings")
bronze_household_df=spark.table("workspace.default.bronze_household_info")

In [0]:
display(bronze_meter_df)

meter_id,household_id,timestamp,units_consumed,_ingestion_time
M001,H001,2026-04-01T00:00:00.000Z,0.41,2026-07-15T13:23:43.674Z
M002,H002,2026-04-01T00:00:00.000Z,0.1,2026-07-15T13:23:43.674Z
M003,H003,2026-04-01T00:00:00.000Z,0.44,2026-07-15T13:23:43.674Z
M004,H004,2026-04-01T00:00:00.000Z,0.23,2026-07-15T13:23:43.674Z
M005,H005,2026-04-01T00:00:00.000Z,6.93,2026-07-15T13:23:43.674Z
M006,H006,2026-04-01T00:00:00.000Z,0.8,2026-07-15T13:23:43.674Z
M007,H007,2026-04-01T00:00:00.000Z,0.47,2026-07-15T13:23:43.674Z
M008,H008,2026-04-01T00:00:00.000Z,0.2,2026-07-15T13:23:43.674Z
M009,H009,2026-04-01T00:00:00.000Z,0.25,2026-07-15T13:23:43.674Z
M010,H010,2026-04-01T00:00:00.000Z,0.25,2026-07-15T13:23:43.674Z


In [0]:
display(bronze_household_df)

household_id,city,house_type,avg_daily_consumption,_ingestion_time
H001,Jaipur,Independent,8,2026-07-15T13:23:55.587Z
H002,Kolkata,Independent,10,2026-07-15T13:23:55.587Z
H003,Jaipur,Villa,12,2026-07-15T13:23:55.587Z
H004,Jaipur,Apartment,15,2026-07-15T13:23:55.587Z
H005,Kolkata,Apartment,7,2026-07-15T13:23:55.587Z
H006,Mumbai,Studio,9,2026-07-15T13:23:55.587Z
H007,Pune,Studio,11,2026-07-15T13:23:55.587Z
H008,Delhi,Studio,14,2026-07-15T13:23:55.587Z
H009,Pune,Independent,6,2026-07-15T13:23:55.587Z
H010,Hyderabad,Independent,13,2026-07-15T13:23:55.587Z


In [0]:
bronze_meter_df.printSchema()

root
 |-- meter_id: string (nullable = true)
 |-- household_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- units_consumed: double (nullable = true)
 |-- _ingestion_time: timestamp (nullable = true)



In [0]:
bronze_household_df.printSchema()

root
 |-- household_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- house_type: string (nullable = true)
 |-- avg_daily_consumption: integer (nullable = true)
 |-- _ingestion_time: timestamp (nullable = true)



In [0]:
bronze_meter_df.count()

1400

In [0]:
bronze_household_df.count()

10

In [0]:
from pyspark.sql.functions import col

bronze_meter_df.select([
    col(c).isNull().cast("int").alias(c)
    for c in bronze_meter_df.columns
]).groupBy().sum().show()

+-------------+-----------------+--------------+-------------------+--------------------+-------------------+
|sum(meter_id)|sum(household_id)|sum(timestamp)|sum(units_consumed)|sum(_ingestion_time)|sum(ingestion_date)|
+-------------+-----------------+--------------+-------------------+--------------------+-------------------+
|            0|                0|             0|                 20|                   0|                  0|
+-------------+-----------------+--------------+-------------------+--------------------+-------------------+



In [0]:
duplicate_count = bronze_meter_df.count() - bronze_meter_df.dropDuplicates(["meter_id", "timestamp"]).count()

print("Duplicate Records:", duplicate_count)

Duplicate Records: 0


In [0]:
bronze_meter_df.filter(col("units_consumed") < 0).show()

+--------+------------+---------+--------------+---------------+
|meter_id|household_id|timestamp|units_consumed|_ingestion_time|
+--------+------------+---------+--------------+---------------+
+--------+------------+---------+--------------+---------------+



In [0]:
bronze_meter_df.filter(col("units_consumed") < 0).count()

0

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:
window_spec = Window.partitionBy("meter_id", "timestamp") \
                    .orderBy("_ingestion_time")

silver_meter_df = bronze_meter_df.withColumn(
    "row_num",
    row_number().over(window_spec)
)

silver_meter_df = silver_meter_df.filter(
    "row_num = 1"
).drop("row_num")

In [0]:
silver_meter_df.count()

1400

In [0]:
from pyspark.sql.functions import col, when

silver_meter_df = silver_meter_df.withColumn(
    "is_valid",
    when(
        (col("units_consumed").isNull()) |
        (col("units_consumed") < 0),
        False
    ).otherwise(True)
)

In [0]:
silver_meter_df.groupBy("is_valid").count().show()

+--------+-----+
|is_valid|count|
+--------+-----+
|    true| 1380|
|   false|   20|
+--------+-----+



## Join Household Information

In this step, we enrich the meter readings by joining them with household information.

This provides additional attributes such as:

- City
- House Type
- Average Daily Consumption

These attributes will be useful for analytics in the Gold layer.

In [0]:
silver_meter_df = silver_meter_df.join(
    bronze_household_df.select(
        "household_id",
        "city",
        "house_type",
        "avg_daily_consumption"
    ),
    on="household_id",
    how="left"
)

In [0]:
display(silver_meter_df)

household_id,meter_id,timestamp,units_consumed,_ingestion_time,is_valid,city,house_type,avg_daily_consumption
H009,M009,2026-04-01T01:00:00.000Z,0.35,2026-07-15T13:23:43.674Z,true,Pune,Independent,6
H010,M010,2026-04-01T03:00:00.000Z,0.1,2026-07-15T13:23:43.674Z,true,Hyderabad,Independent,13
H005,M005,2026-04-01T06:00:00.000Z,4.72,2026-07-15T13:23:43.674Z,true,Kolkata,Apartment,7
H010,M010,2026-04-01T06:00:00.000Z,1.8,2026-07-15T13:23:43.674Z,true,Hyderabad,Independent,13
H001,M001,2026-04-01T09:00:00.000Z,3.87,2026-07-15T13:23:43.674Z,true,Jaipur,Independent,8
H004,M004,2026-04-01T09:00:00.000Z,2.8,2026-07-15T13:23:43.674Z,true,Jaipur,Apartment,15
H002,M002,2026-04-01T10:00:00.000Z,1.06,2026-07-15T13:23:43.674Z,true,Kolkata,Independent,10
H004,M004,2026-04-01T11:00:00.000Z,1.4,2026-07-15T13:23:43.674Z,true,Jaipur,Apartment,15
H002,M002,2026-04-01T13:00:00.000Z,0.68,2026-07-15T13:23:43.674Z,true,Kolkata,Independent,10
H005,M005,2026-04-01T15:00:00.000Z,0.0,2026-07-15T13:23:43.674Z,true,Kolkata,Apartment,7


In [0]:
silver_meter_df.printSchema()

root
 |-- household_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- units_consumed: double (nullable = true)
 |-- _ingestion_time: timestamp (nullable = true)
 |-- is_valid: boolean (nullable = false)
 |-- city: string (nullable = true)
 |-- house_type: string (nullable = true)
 |-- avg_daily_consumption: integer (nullable = true)



In [0]:
silver_meter_df = silver_meter_df.select(
    "meter_id",
    "household_id",
    "timestamp",
    "units_consumed",
    "_ingestion_time",
    "is_valid",
    "city",
    "house_type",
    "avg_daily_consumption"
)

In [0]:
display(silver_meter_df)

meter_id,household_id,timestamp,units_consumed,_ingestion_time,is_valid,city,house_type,avg_daily_consumption
M009,H009,2026-04-01T01:00:00.000Z,0.35,2026-07-15T13:23:43.674Z,true,Pune,Independent,6
M010,H010,2026-04-01T03:00:00.000Z,0.1,2026-07-15T13:23:43.674Z,true,Hyderabad,Independent,13
M005,H005,2026-04-01T06:00:00.000Z,4.72,2026-07-15T13:23:43.674Z,true,Kolkata,Apartment,7
M010,H010,2026-04-01T06:00:00.000Z,1.8,2026-07-15T13:23:43.674Z,true,Hyderabad,Independent,13
M001,H001,2026-04-01T09:00:00.000Z,3.87,2026-07-15T13:23:43.674Z,true,Jaipur,Independent,8
M004,H004,2026-04-01T09:00:00.000Z,2.8,2026-07-15T13:23:43.674Z,true,Jaipur,Apartment,15
M002,H002,2026-04-01T10:00:00.000Z,1.06,2026-07-15T13:23:43.674Z,true,Kolkata,Independent,10
M004,H004,2026-04-01T11:00:00.000Z,1.4,2026-07-15T13:23:43.674Z,true,Jaipur,Apartment,15
M002,H002,2026-04-01T13:00:00.000Z,0.68,2026-07-15T13:23:43.674Z,true,Kolkata,Independent,10
M005,H005,2026-04-01T15:00:00.000Z,0.0,2026-07-15T13:23:43.674Z,true,Kolkata,Apartment,7


In [0]:
silver_meter_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_meter_readings")

In [0]:
%sql
SHOW TABLES;

database,tableName,isTemporary
default,bronze_household_info,false
default,bronze_meter_readings,false
default,customer-incremental,false
default,customer-master,false
default,customer_incremental,false
default,customer_master_delta,false
default,inc,false
default,master,false
default,sample-superstore,false
default,sample_superstore,false


In [0]:
%sql
SELECT * FROM workspace.default.silver_meter_readings;

meter_id,household_id,timestamp,units_consumed,_ingestion_time,is_valid,city,house_type,avg_daily_consumption
M009,H009,2026-04-01T01:00:00.000Z,0.35,2026-07-15T13:23:43.674Z,true,Pune,Independent,6
M010,H010,2026-04-01T03:00:00.000Z,0.1,2026-07-15T13:23:43.674Z,true,Hyderabad,Independent,13
M005,H005,2026-04-01T06:00:00.000Z,4.72,2026-07-15T13:23:43.674Z,true,Kolkata,Apartment,7
M010,H010,2026-04-01T06:00:00.000Z,1.8,2026-07-15T13:23:43.674Z,true,Hyderabad,Independent,13
M001,H001,2026-04-01T09:00:00.000Z,3.87,2026-07-15T13:23:43.674Z,true,Jaipur,Independent,8
M004,H004,2026-04-01T09:00:00.000Z,2.8,2026-07-15T13:23:43.674Z,true,Jaipur,Apartment,15
M002,H002,2026-04-01T10:00:00.000Z,1.06,2026-07-15T13:23:43.674Z,true,Kolkata,Independent,10
M004,H004,2026-04-01T11:00:00.000Z,1.4,2026-07-15T13:23:43.674Z,true,Jaipur,Apartment,15
M002,H002,2026-04-01T13:00:00.000Z,0.68,2026-07-15T13:23:43.674Z,true,Kolkata,Independent,10
M005,H005,2026-04-01T15:00:00.000Z,0.0,2026-07-15T13:23:43.674Z,true,Kolkata,Apartment,7


In [0]:
print("Bronze Records :", bronze_meter_df.count())
print("Silver Records :", silver_meter_df.count())

Bronze Records : 1400
Silver Records : 1400


In [0]:
display(silver_meter_df)

meter_id,household_id,timestamp,units_consumed,_ingestion_time,ingestion_date,is_valid
M001,H001,2026-04-01T00:00:00.000Z,0.41,2026-07-19T06:35:11.923Z,2026-07-19,true
M001,H001,2026-04-01T01:00:00.000Z,0.8,2026-07-19T06:35:11.923Z,2026-07-19,true
M001,H001,2026-04-01T02:00:00.000Z,0.41,2026-07-19T06:35:11.923Z,2026-07-19,true
M001,H001,2026-04-01T03:00:00.000Z,0.96,2026-07-19T06:35:11.923Z,2026-07-19,true
M001,H001,2026-04-01T04:00:00.000Z,0.72,2026-07-19T06:35:11.923Z,2026-07-19,true
M001,H001,2026-04-01T05:00:00.000Z,2.38,2026-07-19T06:35:11.923Z,2026-07-19,true
M001,H001,2026-04-01T06:00:00.000Z,4.82,2026-07-19T06:35:11.923Z,2026-07-19,true
M001,H001,2026-04-01T07:00:00.000Z,4.16,2026-07-19T06:35:11.923Z,2026-07-19,true
M001,H001,2026-04-01T08:00:00.000Z,4.21,2026-07-19T06:35:11.923Z,2026-07-19,true
M001,H001,2026-04-01T09:00:00.000Z,3.87,2026-07-19T06:35:11.923Z,2026-07-19,true


In [0]:
display(spark.table("workspace.default.silver_meter_readings"))

meter_id,household_id,timestamp,units_consumed,_ingestion_time,is_valid,city,house_type,avg_daily_consumption
M009,H009,2026-04-01T01:00:00.000Z,0.35,2026-07-15T13:23:43.674Z,true,Pune,Independent,6
M010,H010,2026-04-01T03:00:00.000Z,0.1,2026-07-15T13:23:43.674Z,true,Hyderabad,Independent,13
M005,H005,2026-04-01T06:00:00.000Z,4.72,2026-07-15T13:23:43.674Z,true,Kolkata,Apartment,7
M010,H010,2026-04-01T06:00:00.000Z,1.8,2026-07-15T13:23:43.674Z,true,Hyderabad,Independent,13
M001,H001,2026-04-01T09:00:00.000Z,3.87,2026-07-15T13:23:43.674Z,true,Jaipur,Independent,8
M004,H004,2026-04-01T09:00:00.000Z,2.8,2026-07-15T13:23:43.674Z,true,Jaipur,Apartment,15
M002,H002,2026-04-01T10:00:00.000Z,1.06,2026-07-15T13:23:43.674Z,true,Kolkata,Independent,10
M004,H004,2026-04-01T11:00:00.000Z,1.4,2026-07-15T13:23:43.674Z,true,Jaipur,Apartment,15
M002,H002,2026-04-01T13:00:00.000Z,0.68,2026-07-15T13:23:43.674Z,true,Kolkata,Independent,10
M005,H005,2026-04-01T15:00:00.000Z,0.0,2026-07-15T13:23:43.674Z,true,Kolkata,Apartment,7


# Silver Layer Summary

## Completed Tasks

- Read Bronze Delta tables
- Performed data quality checks
- Checked null values
- Checked duplicate records
- Checked negative values
- Removed duplicate records
- Created the `is_valid` flag
- Joined household information
- Created the Silver Delta table
- Verified the Silver table

## Silver Output Table

- workspace.default.silver_meter_readings

In [0]:
display(spark.table("workspace.default.silver_meter_readings"))

meter_id,household_id,timestamp,units_consumed,_ingestion_time,is_valid,city,house_type,avg_daily_consumption
M009,H009,2026-04-01T01:00:00.000Z,0.35,2026-07-15T13:23:43.674Z,true,Pune,Independent,6
M010,H010,2026-04-01T03:00:00.000Z,0.1,2026-07-15T13:23:43.674Z,true,Hyderabad,Independent,13
M005,H005,2026-04-01T06:00:00.000Z,4.72,2026-07-15T13:23:43.674Z,true,Kolkata,Apartment,7
M010,H010,2026-04-01T06:00:00.000Z,1.8,2026-07-15T13:23:43.674Z,true,Hyderabad,Independent,13
M001,H001,2026-04-01T09:00:00.000Z,3.87,2026-07-15T13:23:43.674Z,true,Jaipur,Independent,8
M004,H004,2026-04-01T09:00:00.000Z,2.8,2026-07-15T13:23:43.674Z,true,Jaipur,Apartment,15
M002,H002,2026-04-01T10:00:00.000Z,1.06,2026-07-15T13:23:43.674Z,true,Kolkata,Independent,10
M004,H004,2026-04-01T11:00:00.000Z,1.4,2026-07-15T13:23:43.674Z,true,Jaipur,Apartment,15
M002,H002,2026-04-01T13:00:00.000Z,0.68,2026-07-15T13:23:43.674Z,true,Kolkata,Independent,10
M005,H005,2026-04-01T15:00:00.000Z,0.0,2026-07-15T13:23:43.674Z,true,Kolkata,Apartment,7
